1) Kiểm tra price / volume có consistent và full ko?
Loop qua các ngày, qua các stock xem
a) ngày có price và thì có volume và ngược lại
b) số ngày bị missing data có nhiều và có bất thường ko

2) kiểm tra xem price đã được xử lý merge / split chưa.
Mẹo kiểm tra như thế này nhé:
a) tìm xem daily upper bound / lower bound của market là bao nhiêu. Ví dụ, với thị trường Việt Nam, trên hsx là +-7%, trên hnx là +-10%. Bên Mỹ cũng có thể có similar law.
b) em collect một cái array R gồm tất cả các daily returns của tất cả các cổ phiếu trên tất cả các ngày (returns = close_n / close_{n-1} - 1). Plot cái histogram của R ra và xem những giá trị của returns có vi phạm các upper bounds và lower bounds ko. Nếu mà chưa xử lý merge / split thì kiểu gì cũng vi phạm.

# To do
1.   Price / Volume
2.   Price đã được xử lý chưa
3.   Coverage (DATE = union of dates of each stock)




# Setup


In [ ]:
# !git clone https://github.com/nhaTahn/data_stock_market.git

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import datetime

# Const

In [ ]:
market_folder = '/content/data_stock_market/data/VN'
output_base = '/content/data_stock_market/data/VN_Classified'

In [ ]:
#Ref: https://vnexpress.net/bien-do-giao-dich-co-phieu-tren-cac-san-chung-khoan-4315116.html#:~:text=Bi%C3%AAn%20%C4%91%E1%BB%99%20gi%C3%A1%2C%20HoSE%2C%20HNX%2C%20UpCOM%20;,tr%E1%BB%9F%20l%E1%BA%A1i%20sau%20khi%20b%E1%BB%8B%20t%E1%BA%A1m%20ng%E1%BB%ABng

# ngày thường
normal_limit_map = {
    "HSX": 0.07,
    "HNX": 0.10,
    "UPCOM": 0.15,
}

# map biên độ đặc biệt
# Cổ phiếu mới niêm yết trong ngày đầu tiên hoặc được giao dịch trở lại sau khi bị tạm ngừng giao dịch trên 25 ngày
# Cổ phiếu trong ngày giao dịch không hưởng quyền để trả cổ tức hoặc thưởng bằng cổ phiếu quỹ cho cổ đông hiện hữu
special_limit_map = {
    "HSX": 0.20,
    "HNX": 0.30,
    "UPCOM": 0.40,
}

In [ ]:
vn100_list = [
    "ACB", "ANV", "BCM", "BID", "BMP", "BSI", "BVH", "BWE", "CII", "CMG",
    "CTD", "CTG", "CTR", "CTS", "DBC", "DCM", "DGC", "DGW", "DIG", "DPM",
    "DSE", "DXG", "DXS", "EIB", "EVF", "FPT", "FRT", "FTS", "GAS", "GEE",
    "GEX", "GMD", "GVR", "HAG", "HCM", "HDB", "HDC", "HDG", "HHV", "HPG",
    "HSG", "HT1", "IMP", "KBC", "KDC", "KDH", "KOS", "LPB", "MBB", "MSB",
    "MSN", "MWG", "NAB", "NKG", "NLG", "NT2", "OCB", "PAN", "PC1", "PDR",
    "PHR", "PLX", "PNJ", "POW", "PPC", "PTB", "PVD", "PVT", "REE", "SAB",
    "SBT", "SCS", "SHB", "SIP", "SJS", "SSB", "SSI", "STB", "SZC", "TCB",
    "TCH", "TLG", "TPB", "VCB", "VCG", "VCI", "VGC", "VHC", "VHM", "VIB",
    "VIC", "VIX", "VJC", "VND", "VNM", "VPB", "VPI", "VRE", "VSC", "VTP"
]

In [ ]:
# Date	code	high	low	open	close	adjust	volume_match	value_match
col_mapping = {'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume_match'}

In [ ]:
time_train_min = '2020-01-01'
time_train_max = '2023-12-31'

time_test_min = '2022-09-01'
time_test_max = datetime.date.today().isoformat()

# Sanity data:
Hiện đang focus cho data của VN100 (Hose - HSX)


In [ ]:
# READ AND COMBINE DATA
all_files = glob.glob(os.path.join(market_folder, "*.csv"))
print(f"Start processing {len(all_files)} data files...\n")

df_list = []
for file in all_files:
    # Read each CSV file
    df = pd.read_csv(file)
    # Extract stock code from filename (e.g., FPT.csv -> FPT)
    stock_code = os.path.basename(file).replace('.csv', '')
    df['code_name'] = stock_code
    df_list.append(df)

# Combine all individual dataframes into one large dataframe for efficient processing
combined_df = pd.concat(df_list, ignore_index=True)

In [ ]:
def check_data_consistency(df, price_col="close", vol_col="volume_match", ticker_col="code", anomaly_threshold=5.0):
    """
    Checks a stock dataframe for data consistency (Price vs Volume)
    and missing data anomalies.

    Parameters:
        df (pd.DataFrame): The combined dataframe containing stock data.
        price_col (str): The column name for price data.
        vol_col (str): The column name for volume data.
        ticker_col (str): The column name for the stock ticker/code.
        anomaly_threshold (float): The threshold % for missing data to be flagged as an anomaly.
        => default = 5%: Nếu một mã thiếu 5% dữ liệu, tức là nó bị trống khoảng 12.5 ngày.
          - 1-2%: For AI
          - 5%: Standard
    Returns:
        tuple: (inconsistent_df, anomalies_df)
               - inconsistent_df: Rows where price is missing but volume exists, or vice versa.
               - anomalies_df: Summary dataframe of stocks exceeding the missing data threshold.
    """
    print(f"Size {df.size}")
    print("\n" + "="*50)
    print("=== DATA CONSISTENCY & COMPLETENESS CHECK ===")

    # 1. Ensure required columns exist before proceeding
    required_cols = [price_col, vol_col, ticker_col]
    if not all(col in df.columns for col in required_cols):
        print(f"Error: DataFrame is missing one or more required columns: {required_cols}")
        print(f"Current columns in your DataFrame are: {df.columns.tolist()}") # Thêm dòng này để dễ debug
        return None, None

    # ---------------------------------------------------------
    # A) CONSISTENCY CHECK (Price vs Volume)
    # ---------------------------------------------------------
    print("\n--- A) Consistency Check ---")

    # Condition: Price is NaN BUT Volume has data, OR Price has data BUT Volume is NaN
    is_inconsistent = (
        df[price_col].notna() & df[vol_col].isna()
    ) | (
        df[price_col].isna() & df[vol_col].notna()
    )

    inconsistent_df = df[is_inconsistent].copy()
    print(f"Total inconsistent rows found across all stocks: {len(inconsistent_df)}")

    if not inconsistent_df.empty:
        # Summarize which stocks have the most issues
        inconsistent_summary = inconsistent_df.groupby(ticker_col).size().reset_index(name='Inconsistent_Days')
        inconsistent_summary = inconsistent_summary.sort_values(by='Inconsistent_Days', ascending=False)
        print("Top stocks with inconsistent data:")
        print(inconsistent_summary.head(10).to_string(index=False))

    # ---------------------------------------------------------
    # B) MISSING DATA & ANOMALY CHECK
    # ---------------------------------------------------------
    print(f"\n--- B) Missing Data Check (Threshold: > {anomaly_threshold}%) ---")

    # Group by stock code to calculate total rows and missing values
    grouped = df.groupby(ticker_col)

    stats = grouped.agg(
        Total_Days=(ticker_col, 'size'),
        Missing_Price=(price_col, lambda x: x.isna().sum()),
        Missing_Volume=(vol_col, lambda x: x.isna().sum())
    ).reset_index()

    # Find the maximum missing days (between price and volume) to calculate worst-case %
    stats['Max_Missing'] = stats[['Missing_Price', 'Missing_Volume']].max(axis=1)
    stats['Missing_Percent'] = (stats['Max_Missing'] / stats['Total_Days']) * 100

    # Filter anomalies exceeding our defined threshold
    anomalies_df = stats[stats['Missing_Percent'] > anomaly_threshold].copy()

    print(f"Found {len(anomalies_df)} stock(s) with abnormal missing data.")

    if not anomalies_df.empty:
        anomalies_df = anomalies_df.sort_values(by='Missing_Percent', ascending=False)
        anomalies_df['Missing_Percent'] = anomalies_df['Missing_Percent'].round(2)

        display_cols = [ticker_col, 'Total_Days', 'Max_Missing', 'Missing_Percent']
        print("Top missing data anomalies:")
        print(anomalies_df[display_cols].head(10).to_string(index=False))

    return inconsistent_df, anomalies_df

In [ ]:
inconsistent_data, missing_anomalies = check_data_consistency(
    df=combined_df,
    anomaly_threshold=5.0
)

In [ ]:
inconsistent_data, missing_anomalies = check_data_consistency(
    df=combined_df,
    anomaly_threshold=2.0
)

In [ ]:
def check_price_split_merge(df, price_col="adjust", limit=0.15, start_date="2020-01-01"):
    """
    Calculates daily returns, plots a histogram, and checks for boundary violations
    to verify if stock prices have been adjusted for splits and merges.

    Note: We use 'adjust' column instead of 'close' to account for corporate actions.
    We set default limit to 0.15 to account for VN100 stocks that historically traded on UPCOM.
    """
    print("\n" + "="*50)
    print(f"--- CHECKING PRICE ADJUSTMENTS (SPLIT/MERGE) SINCE {start_date}---")

    # Create a copy to avoid modifying the original dataframe
    df_plot = df.copy()

    # Step 1: Ensure data is properly sorted by Ticker (code) and Date
    df_plot['Date'] = pd.to_datetime(df_plot['Date'])
    df_plot = df_plot.sort_values(by=['code', 'Date'])

    # Step 2: Calculate Daily Returns (R) using the ADJUSTED price
    df_plot['Return'] = df_plot.groupby('code')[price_col].pct_change()

    # Filter from start_time
    start_dt = pd.to_datetime(start_date)
    df_plot = df_plot[df_plot['Date'] >= start_dt].copy()

    # Drop NaN values that typically appear on the first trading day of each stock
    returns_array = df_plot['Return'].dropna()

    # Step 3: Plot the Histogram
    plt.figure(figsize=(12, 6))
    # Plot histogram with 300 bins
    plt.hist(returns_array, bins=300, color='mediumseagreen', edgecolor='black', alpha=0.7)
    plt.xlim(-0.16, 0.16)

    # Draw Upper and Lower bound limit lines
    plt.axvline(x=0.07, color='red', linestyle='dotted', linewidth=1.5, label='HSX Limit (+/- 7%)')
    plt.axvline(x=-0.07, color='red', linestyle='dotted', linewidth=1.5)
    plt.axvline(x=limit, color='darkred', linestyle='dashed', linewidth=2, label=f'Historical/UPCOM Bound (+/- {limit*100}%)')
    plt.axvline(x=-limit, color='darkred', linestyle='dashed', linewidth=2)

    # Chart decorations
    plt.title(f"Distribution of Daily Returns (R) using '{price_col}' Prices")
    plt.xlabel('Daily Returns (R)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.show()

    # Step 4: Filter and display boundary violations
    # Allow a small margin of error (0.005) for rounding differences
    violations = df_plot[df_plot['Return'].abs() > (limit + 0.005)]

    print(f"Total return records analyzed: {len(returns_array)}")
    print(f"Number of violations exceeding {limit*100}% threshold: {len(violations)}")

    if len(violations) > 0:
        print("\nTop boundary violations (Check if these are First-Day Trading events):")
        worst_violations = violations.sort_values(by='Return', key=abs, ascending=False)
        # Display relevant columns to help you verify
        print(worst_violations[['code', 'Date', 'close', 'adjust', 'Return']].head())
    else:
        print("\nGreat news! No major violations found. The price data appears to be properly adjusted.")

In [ ]:
# Execute the function with your combined dataset
check_price_split_merge(combined_df, price_col="adjust", limit=normal_limit_map['UPCOM'])

In [ ]:
# Execute the function with your combined dataset
check_price_split_merge(combined_df, price_col="adjust", limit=special_limit_map['UPCOM'])

In [ ]:
check_price_split_merge(combined_df, price_col="adjust", limit=normal_limit_map['HSX'])

In [ ]:
check_price_split_merge(combined_df, price_col="adjust", limit=normal_limit_map['HNX'])


In [ ]:
def check_data_coverage(df, date_col="Date", ticker_col="code_name"):
    """
    Calculates the date coverage ratio for each stock compared to the
    overall market timeline (union of all dates).

    Parameters:
        df (pd.DataFrame): The combined dataframe containing all stocks.
        date_col (str): The column name for dates.
        ticker_col (str): The column name for the stock ticker.

    Returns:
        pd.DataFrame: A summary dataframe containing the coverage ratio for each stock.
    """
    print("\n" + "="*50)
    print("=== DATA COVERAGE CHECK ===")

    # 1. DATE = union of dates of each stock
    df[date_col] = pd.to_datetime(df[date_col])

    master_dates = df[date_col].dropna().unique()
    total_market_days = len(master_dates)

    market_start_date = df[date_col].min().strftime('%Y-%m-%d')
    market_end_date = df[date_col].max().strftime('%Y-%m-%d')

    # Print the requested information
    print(f"Market Start Date: {market_start_date}")
    print(f"Market End Date:   {market_end_date}")
    print(f"Total unique trading days in the market (Union DATE): {total_market_days} days\n")

    # 2. Count date_stock for each stock
    coverage_df = df.groupby(ticker_col).agg(
        stock_days=(date_col, 'nunique'),
        first_trading_day=(date_col, 'min')
    ).reset_index()

    coverage_df['first_trading_day'] = coverage_df['first_trading_day'].dt.strftime('%Y-%m-%d')
    #Calculate the coverage ratio (date_stock / DATE)
    coverage_df['coverage_ratio'] = coverage_df['stock_days'] / total_market_days
    coverage_df['coverage_percent'] = (coverage_df['coverage_ratio'] * 100).round(2)

    # Sort from lowest coverage to highest to easily spot problematic stocks
    coverage_df = coverage_df.sort_values(by='coverage_percent', ascending=True)

    # --- Print Summary Report ---
    print("\n--- Coverage Summary ---")

    # Flag stocks with less than 50% coverage as an example
    LOW_COVERAGE_THRESHOLD = 50.0
    low_coverage_stocks = coverage_df[coverage_df['coverage_percent'] < LOW_COVERAGE_THRESHOLD]

    print(f"Found {len(low_coverage_stocks)} stock(s) with less than {LOW_COVERAGE_THRESHOLD}% coverage.")

    if len(low_coverage_stocks) > 0:
        print(f"\nTop {len(low_coverage_stocks)} stocks with the LOWEST coverage:")

        display_df = coverage_df.copy()
        display_df['coverage_percent'] = display_df['coverage_percent'].astype(str) + '%'

        cols_to_display = [ticker_col, 'first_trading_day', 'stock_days', 'coverage_percent']
        print(display_df[cols_to_display].head(len(low_coverage_stocks)).to_string(index=False))

    return coverage_df
    return coverage_df

In [ ]:
coverage_report = check_data_coverage(combined_df)

# Filter for train

Data coverage:
1) during 2020-now: coverage >= 70%
2) latest date ~ now

In [ ]:
def rigorous_sanity_check_and_split(df, date_col="Date", ticker_col="code_name", price_col="adjust"):
    """
    Performs a rigorous sanity check based on specific timeframes, coverage >= 70%,
    latest date condition, and splits data into train/test sets.
    Plots the distribution of returns with min, max, and quantiles.
    """
    print("\n" + "="*50)
    print("=== RIGOROUS SANITY CHECK & TRAIN/TEST SPLIT ===")

    # 1. Ensure Date is datetime format
    df_main = df.copy()
    df_main[date_col] = pd.to_datetime(df_main[date_col])

    market_now = pd.to_datetime(time_test_max) # Now
    start_2020 = pd.to_datetime(time_train_min)
    print(f"Market 'Now' defined as: {market_now.strftime('%Y-%m-%d')}")

    # ---------------------------------------------------------
    # A) FILTER STOCKS BY COVERAGE (2020 to Now)
    # ---------------------------------------------------------
    print(f"\n--- A) Checking Coverage ({time_train_min} - {time_test_max}) ---")

    # Filter data to the [2020 - now] window
    df_2020_now = df_main[(df_main[date_col] >= start_2020) & (df_main[date_col] <= market_now)]
    total_market_days = df_2020_now[date_col].nunique()

    # Calculate coverage and latest date for each stock
    coverage_stats = df_2020_now.groupby(ticker_col).agg(
        stock_days=(date_col, 'nunique'),
        first_date=(date_col, 'min'),
        latest_date=(date_col, 'max')
    ).reset_index()

    coverage_stats['coverage_pct'] = coverage_stats['stock_days'] / total_market_days

    # Condition 1: coverage >= 70% (0.70)
    # Condition 2: latest date ~ now (Within 30 days of market_now)
    tolerance_days = pd.Timedelta(days=30)

    valid_mask = (coverage_stats['coverage_pct'] >= 0.70) & (coverage_stats['latest_date'] >= (market_now - tolerance_days))

    valid_stocks_df = coverage_stats[valid_mask]
    invalid_stocks_df = coverage_stats[~valid_mask].copy()

    valid_tickers = valid_stocks_df[ticker_col].tolist()
    print(f"Total market days (2020-Now): {total_market_days}")
    print(f"Stocks passed condition (Coverage >= 70% & Active recently): {len(valid_tickers)} / {len(coverage_stats)}")

    print(f"\n* INVALID DATA (Found {len(invalid_stocks_df)} stocks):")
    if len(invalid_stocks_df) > 0:
        invalid_stocks_df['coverage_pct'] = (invalid_stocks_df['coverage_pct'] * 100).round(2).astype(str) + '%'
        invalid_stocks_df['first_date'] = invalid_stocks_df['first_date'].dt.strftime('%Y-%m-%d')
        invalid_stocks_df['latest_date'] = invalid_stocks_df['latest_date'].dt.strftime('%Y-%m-%d')
        invalid_stocks_df = invalid_stocks_df.sort_values(by='coverage_pct')
        print(invalid_stocks_df.to_string(index=False))
    else:
        print("All stocks are valid!")

    # Filter main dataset to only keep these valid stocks
    df_valid = df_main[df_main[ticker_col].isin(valid_tickers)].copy()

    # ---------------------------------------------------------
    # B) TIME USE SPLIT (Train & Test)
    # ---------------------------------------------------------
    print("\n--- B) Creating Train & Test Sets ---")

    # train_set = [2020-01-01 to 2023-12-31]
    train_set = df_valid[(df_valid[date_col] >= time_train_min) & (df_valid[date_col] <= time_train_max)].copy()

    # test_set = [2022-09-01 to now]
    test_set = df_valid[(df_valid[date_col] >= time_test_min) & (df_valid[date_col] <= time_test_max)].copy()

    print(f"Train Set Shape [2020 - 2023]: {train_set.shape}")
    print(f"Test Set Shape  [09/2022 - Now]: {test_set.shape}")

    # ---------------------------------------------------------
    # C) STATISTICAL SUMMARY & HISTOGRAM (Daily Returns R)
    # ---------------------------------------------------------
    print("\n--- C) Returns Distribution Analysis ---")

    # Calculate R for the valid dataset
    df_valid = df_valid.sort_values(by=[ticker_col, date_col])
    df_valid['R'] = df_valid.groupby(ticker_col)[price_col].pct_change()

    R_array = df_valid['R'].dropna()

    # Calculate statistics
    r_min = R_array.min()
    r_max = R_array.max()
    qt_05 = R_array.quantile(0.05)
    qt_95 = R_array.quantile(0.95)

    # Print the requested values
    print(f"MIN:      {r_min:.6f}")
    print(f"MAX:      {r_max:.6f}")
    print(f"QT(0.05): {qt_05:.6f}")
    print(f"QT(0.95): {qt_95:.6f}")

    # Plotting
    plt.figure(figsize=(10, 5))

    # Filter extreme outliers just for a cleaner visual plot (e.g., zoom in to +/- 15%)
    # but the calculations above used the RAW data.
    plot_data = R_array.clip(lower=-0.15, upper=0.15)

    plt.hist(plot_data, bins=100, color='royalblue', edgecolor='black', alpha=0.7)

    # Add vertical lines for quantiles
    plt.axvline(qt_05, color='red', linestyle='dashed', linewidth=2, label=f'5th Pct: {qt_05:.4f}')
    plt.axvline(qt_95, color='green', linestyle='dashed', linewidth=2, label=f'95th Pct: {qt_95:.4f}')

    plt.title('Histogram of Daily Returns (R) for Valid Stocks (2020-Now)')
    plt.xlabel('Daily Return (R)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.show()

    return train_set, test_set


In [ ]:
train_df, test_df = rigorous_sanity_check_and_split(combined_df, price_col="adjust")

# Trust-first clean pipeline

Notebook này đã được nối với pipeline `build_vn_quality_dataset.py` để ưu tiên dữ liệu sạch và truy vết được lý do loại bỏ.

Các output chính:
- `vn_input_quality_overview.csv`: trạng thái tập đầu vào trước clean
- `vn_ticker_quality_summary.csv`: chất lượng theo từng mã
- `vn_row_quality_flags.csv`: cờ chất lượng theo từng dòng
- `vn_gold_balanced.csv`, `vn_gold_model_strict.csv`, `vn_gold_trust_max.csv`: ba hướng clean


In [ ]:
from pathlib import Path

quality_dir = Path('/Users/lap15111/Documents/research-paper/data_stock_market/data/assets/data_info_vn/history')
input_overview = pd.read_csv(quality_dir / 'vn_input_quality_overview.csv')
ticker_quality = pd.read_csv(quality_dir / 'vn_ticker_quality_summary.csv')
profile_summary = pd.read_csv(quality_dir / 'vn_clean_profile_summary.csv')
row_flags = pd.read_csv(quality_dir / 'vn_row_quality_flags.csv', parse_dates=['Date'])

print('Input dataset overview:')
display(input_overview)

print('Clean profile summary:')
display(profile_summary)


In [ ]:
print('Top 15 tickers with lowest quality score:')
display(
    ticker_quality.sort_values(['quality_score', 'coverage_pct']).head(15)[
        ['code', 'coverage_pct', 'days_since_latest', 'hard_issue_rows', 'imputed_rows',
         'close_violation_7pct_rows', 'adjust_jump_155pct_rows', 'quality_score', 'quality_band']
    ]
)

print('Sample hard issues / rows to inspect first:')
display(
    row_flags[
        row_flags['has_hard_issue']
        | row_flags['close_bound_violation_7pct']
        | row_flags['adjust_event_jump_155pct']
    ][['code', 'Date', 'open', 'high', 'low', 'close', 'adjust', 'volume_match',
       'ohlc_invalid', 'value_match_imputed', 'close_return', 'adjust_return']]
    .sort_values(['Date', 'code'], ascending=[False, True])
    .head(30)
)


## How to use each clean profile

- `balanced`: giữ breadth tối đa cho phân tích khám phá, chỉ loại hard issues và jump lớn rõ ràng.
- `model_strict`: phù hợp train baseline cho RF/LSTM, loại thêm imputed rows và buffer quanh event.
- `trust_max`: ưu tiên độ tin cậy cao nhất, chấp nhận hy sinh coverage và event-heavy rows.

Nếu mục tiêu là train model tin cậy nhất, nên bắt đầu từ `vn_gold_trust_max.csv`.


# Colab-ready trust-first cleaning

Section này chạy độc lập ngay trong Colab, không phụ thuộc file script ngoài notebook.
Nó sẽ:
- show trạng thái dữ liệu đầu vào trước clean
- gắn cờ chất lượng từng dòng
- chấm điểm chất lượng từng mã
- export 3 tập clean: `balanced`, `model_strict`, `trust_max`


In [ ]:
from dataclasses import dataclass
from pathlib import Path

# Nếu chạy trên Colab sau khi git clone repo vào /content/data_stock_market thì block này sẽ tự nhận đúng path.
project_root = Path(market_folder).resolve().parents[1] if 'market_folder' in globals() else Path('/content/data_stock_market')
quality_output_dir = project_root / 'data' / 'assets' / 'data_info_vn' / 'history'
quality_output_dir.mkdir(parents=True, exist_ok=True)

TRAIN_START_DATE = '2020-01-01'
RECENT_ACTIVE_TOLERANCE_DAYS = 30

@dataclass(frozen=True)
class CleanProfile:
    name: str
    min_coverage: float
    max_close_return_abs: float
    max_adjust_return_abs: float
    drop_imputed_value_match: bool
    drop_neighbors_around_events: bool
    recent_active_tolerance_days: int = RECENT_ACTIVE_TOLERANCE_DAYS

CLEAN_PROFILES = [
    CleanProfile('balanced', 0.70, 0.15, 0.20, False, False),
    CleanProfile('model_strict', 0.90, 0.10, 0.16, True, True),
    CleanProfile('trust_max', 0.95, 0.075, 0.155, True, True),
]

def load_vn_data_for_quality(market_folder):
    all_files = glob.glob(os.path.join(market_folder, '*.csv'))
    frames = []
    for file in sorted(all_files):
        df = pd.read_csv(file)
        df['source_file'] = os.path.basename(file)
        frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    combined['Date'] = pd.to_datetime(combined['Date'])
    combined = combined.sort_values(['code', 'Date']).reset_index(drop=True)
    combined['row_id'] = np.arange(len(combined))
    return combined

def compute_row_quality_flags(df):
    out = df.copy()
    required_cols = ['Date', 'code', 'open', 'high', 'low', 'close', 'adjust', 'volume_match']
    out['missing_required'] = out[required_cols].isna().any(axis=1)
    out['duplicate_code_date'] = out.duplicated(subset=['code', 'Date'], keep=False)

    for col in ['open', 'high', 'low', 'close', 'adjust', 'volume_match']:
        out[col] = pd.to_numeric(out[col], errors='coerce')

    out['negative_price'] = (out[['open', 'high', 'low', 'close', 'adjust']] < 0).any(axis=1)
    out['negative_volume'] = out['volume_match'] < 0
    out['ohlc_invalid'] = (
        (out['high'] < out[['open', 'close', 'low']].max(axis=1))
        | (out['low'] > out[['open', 'close', 'high']].min(axis=1))
        | (out['high'] < out['low'])
    )

    if 'value_match_imputed' in out.columns:
        out['value_match_imputed'] = pd.to_numeric(out['value_match_imputed'], errors='coerce').fillna(0).astype(int)
    else:
        out['value_match_imputed'] = 0

    out['close_return'] = out.groupby('code')['close'].pct_change()
    out['adjust_return'] = out.groupby('code')['adjust'].pct_change()
    out['close_bound_violation_7pct'] = out['close_return'].abs() > 0.075
    out['close_bound_violation_10pct'] = out['close_return'].abs() > 0.10
    out['close_bound_violation_15pct'] = out['close_return'].abs() > 0.15
    out['adjust_event_jump_155pct'] = out['adjust_return'].abs() > 0.155
    out['adjust_event_jump_20pct'] = out['adjust_return'].abs() > 0.20

    out['has_hard_issue'] = (
        out['missing_required']
        | out['duplicate_code_date']
        | out['negative_price']
        | out['negative_volume']
        | out['ohlc_invalid']
    )
    return out

def compute_ticker_quality(row_flags, train_start='2020-01-01'):
    recent = row_flags[row_flags['Date'] >= pd.Timestamp(train_start)].copy()
    market_now = recent['Date'].max()
    total_market_days = recent['Date'].nunique()

    ticker_quality = recent.groupby('code').agg(
        stock_days=('Date', 'nunique'),
        first_date=('Date', 'min'),
        latest_date=('Date', 'max'),
        hard_issue_rows=('has_hard_issue', 'sum'),
        imputed_rows=('value_match_imputed', 'sum'),
        close_violation_7pct_rows=('close_bound_violation_7pct', 'sum'),
        close_violation_10pct_rows=('close_bound_violation_10pct', 'sum'),
        close_violation_15pct_rows=('close_bound_violation_15pct', 'sum'),
        adjust_jump_155pct_rows=('adjust_event_jump_155pct', 'sum'),
        adjust_jump_20pct_rows=('adjust_event_jump_20pct', 'sum')
    ).reset_index()

    ticker_quality['coverage_pct'] = ticker_quality['stock_days'] / total_market_days
    ticker_quality['days_since_latest'] = (market_now - ticker_quality['latest_date']).dt.days
    ticker_quality['is_recently_active'] = ticker_quality['days_since_latest'] <= RECENT_ACTIVE_TOLERANCE_DAYS
    ticker_quality['quality_score'] = (
        100
        - ticker_quality['hard_issue_rows'] * 10
        - ticker_quality['imputed_rows'] * 0.05
        - ticker_quality['close_violation_7pct_rows'] * 1.5
        - ticker_quality['adjust_jump_155pct_rows'] * 2.5
        - (1 - ticker_quality['coverage_pct']) * 50
    ).clip(lower=0)

    ticker_quality['quality_band'] = pd.cut(
        ticker_quality['quality_score'],
        bins=[-1, 60, 80, 90, 101],
        labels=['review', 'usable', 'good', 'high_trust']
    )
    return ticker_quality.sort_values(['quality_score', 'coverage_pct', 'code'], ascending=[False, False, True]).reset_index(drop=True)

def compute_neighbor_event_flags(df, event_mask):
    event_series = event_mask.astype(bool)
    prev_event = event_series.groupby(df['code']).shift(1).eq(True)
    next_event = event_series.groupby(df['code']).shift(-1).eq(True)
    return event_series | prev_event | next_event

def build_overview_table(row_flags, ticker_quality):
    metrics = [
        ('total_rows', len(row_flags)),
        ('total_tickers', row_flags['code'].nunique()),
        ('total_market_days', row_flags['Date'].nunique()),
        ('market_start', row_flags['Date'].min().date().isoformat()),
        ('market_end', row_flags['Date'].max().date().isoformat()),
        ('missing_required_rows', int(row_flags['missing_required'].sum())),
        ('duplicate_code_date_rows', int(row_flags['duplicate_code_date'].sum())),
        ('negative_price_rows', int(row_flags['negative_price'].sum())),
        ('negative_volume_rows', int(row_flags['negative_volume'].sum())),
        ('ohlc_invalid_rows', int(row_flags['ohlc_invalid'].sum())),
        ('value_match_imputed_rows', int(row_flags['value_match_imputed'].sum())),
        ('close_violation_7pct_rows', int(row_flags['close_bound_violation_7pct'].sum())),
        ('close_violation_10pct_rows', int(row_flags['close_bound_violation_10pct'].sum())),
        ('close_violation_15pct_rows', int(row_flags['close_bound_violation_15pct'].sum())),
        ('adjust_jump_155pct_rows', int(row_flags['adjust_event_jump_155pct'].sum())),
        ('adjust_jump_20pct_rows', int(row_flags['adjust_event_jump_20pct'].sum())),
        ('high_trust_tickers', int((ticker_quality['quality_band'] == 'high_trust').sum())),
        ('good_or_better_tickers', int(ticker_quality['quality_band'].isin(['good', 'high_trust']).sum())),
    ]
    return pd.DataFrame(metrics, columns=['metric', 'value'])

def build_clean_dataset(row_flags, ticker_quality, profile):
    valid_tickers = ticker_quality[
        (ticker_quality['coverage_pct'] >= profile.min_coverage)
        & (ticker_quality['days_since_latest'] <= profile.recent_active_tolerance_days)
    ]['code']

    clean = row_flags[row_flags['code'].isin(valid_tickers)].copy()
    clean['profile_event'] = (
        clean['close_return'].abs().gt(profile.max_close_return_abs)
        | clean['adjust_return'].abs().gt(profile.max_adjust_return_abs)
    ).fillna(False)

    if profile.drop_neighbors_around_events:
        clean['drop_due_to_event_buffer'] = compute_neighbor_event_flags(clean, clean['profile_event'])
    else:
        clean['drop_due_to_event_buffer'] = clean['profile_event']

    keep_mask = ~clean['has_hard_issue']
    if profile.drop_imputed_value_match:
        keep_mask &= clean['value_match_imputed'].eq(0)
    keep_mask &= ~clean['drop_due_to_event_buffer']

    clean = clean[keep_mask].copy()
    clean['target_next_adjust_return'] = clean.groupby('code')['adjust'].shift(-1) / clean['adjust'] - 1
    clean['target_next_close_return'] = clean.groupby('code')['close'].shift(-1) / clean['close'] - 1
    clean['target_next_adjust_price'] = clean.groupby('code')['adjust'].shift(-1)
    clean['target_next_close_price'] = clean.groupby('code')['close'].shift(-1)
    clean['clean_profile'] = profile.name
    return clean


In [ ]:
vn_quality_df = load_vn_data_for_quality(market_folder)
row_flags = compute_row_quality_flags(vn_quality_df)
ticker_quality = compute_ticker_quality(row_flags, train_start=TRAIN_START_DATE)
input_overview = build_overview_table(row_flags, ticker_quality)

clean_datasets = {}
profile_rows = []
for profile in CLEAN_PROFILES:
    clean_df = build_clean_dataset(row_flags, ticker_quality, profile)
    clean_datasets[profile.name] = clean_df
    profile_rows.append({
        'profile': profile.name,
        'rows_kept': len(clean_df),
        'tickers_kept': clean_df['code'].nunique(),
        'date_start': clean_df['Date'].min().date().isoformat() if not clean_df.empty else None,
        'date_end': clean_df['Date'].max().date().isoformat() if not clean_df.empty else None,
        'row_retention_pct': round(len(clean_df) / len(row_flags) * 100, 2),
        'ticker_retention_pct': round(clean_df['code'].nunique() / row_flags['code'].nunique() * 100, 2),
        'rows_with_target': int(clean_df['target_next_adjust_return'].notna().sum()),
    })

profile_summary = pd.DataFrame(profile_rows)

display(input_overview)
display(profile_summary)


In [ ]:
print('Top 15 tickers cần review trước:')
display(
    ticker_quality.sort_values(['quality_score', 'coverage_pct']).head(15)[
        ['code', 'coverage_pct', 'days_since_latest', 'hard_issue_rows', 'imputed_rows',
         'close_violation_7pct_rows', 'adjust_jump_155pct_rows', 'quality_score', 'quality_band']
    ]
)

print('Sample rows có vấn đề để kiểm tra đầu vào:')
display(
    row_flags[
        row_flags['has_hard_issue']
        | row_flags['close_bound_violation_7pct']
        | row_flags['adjust_event_jump_155pct']
    ][['code', 'Date', 'open', 'high', 'low', 'close', 'adjust', 'volume_match',
       'ohlc_invalid', 'value_match_imputed', 'close_return', 'adjust_return']]
    .sort_values(['Date', 'code'], ascending=[False, True])
    .head(50)
)


In [ ]:
input_overview.to_csv(quality_output_dir / 'vn_input_quality_overview.csv', index=False)
ticker_quality.to_csv(quality_output_dir / 'vn_ticker_quality_summary.csv', index=False)
row_flags.to_csv(quality_output_dir / 'vn_row_quality_flags.csv', index=False)
profile_summary.to_csv(quality_output_dir / 'vn_clean_profile_summary.csv', index=False)
for profile_name, clean_df in clean_datasets.items():
    clean_df.to_csv(quality_output_dir / f'vn_gold_{profile_name}.csv', index=False)

print(f'Exported outputs to: {quality_output_dir}')
